# Domain-Adaptive Colorization — Pipeline Runner

Runs `main.py` end to end: Grounded-SAM detection/routing → per-domain DDColor experts → soft blending → RGB reconstruction.

**Kaggle settings:** GPU accelerator (T4 x2 or P100) and **Internet ON**.

In [ ]:
# Cell 1: clone the project (branch B)
from pathlib import Path

REPO_DIR = Path("/kaggle/working/domain-adaptive-colorization")
if not REPO_DIR.exists():
    !git clone -b B https://github.com/zaeem886/domain-adaptive-colorization.git {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
# Cell 2: install all dependencies upfront safely
!pip install -q --no-deps lpips
!pip install -q segment-anything transformers timm einops scikit-image scipy opencv-python-headless huggingface-hub pyyaml tqdm pytest matplotlib pandas

In [ ]:
# Cell 3: run the test suite (no GPU, torch, cv2, or model weights required)
!PYTHONPATH=. python -m pytest -q tests

In [ ]:
# Cell 4: clone DDColor and download the pretrained DDColor-T/L checkpoints
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

DDCOLOR_DIR = Path("/kaggle/working/DDColor")
if not DDCOLOR_DIR.exists():
    !git clone --depth 1 https://github.com/piddnad/DDColor.git {DDCOLOR_DIR}

CHECKPOINT_T = hf_hub_download("piddnad/ddcolor_paper_tiny", "pytorch_model.bin")
CHECKPOINT_L = hf_hub_download("piddnad/ddcolor_paper", "pytorch_model.bin")
CHECKPOINT = CHECKPOINT_T  # used by the single-image demo in Cell 5
print("DDColor:", DDCOLOR_DIR)
print("DDColor-T checkpoint:", CHECKPOINT_T)
print("DDColor-L checkpoint:", CHECKPOINT_L)

In [ ]:
# Cell 5: run the pipeline
IMAGE_PATH = "pics/dog.jpeg"          # change to your own image
OUTPUT_PATH = "outputs/kaggle_colorized.png"

!python main.py \
    --image {IMAGE_PATH} \
    --output {OUTPUT_PATH} \
    --single-checkpoint {CHECKPOINT} \
    --ddcolor-path {DDCOLOR_DIR} \
    --sam-checkpoint sam_vit_b.pth \
    --device cuda \
    --save-domain-map \
    --visualize \
    --stages-dir outputs/stages

In [ ]:
# Cell 6: display the intermediate pipeline stages
import os
from IPython.display import Image as IPImage, display

STAGES_DIR = "outputs/stages"
stages = [
    ("1. Decomposition", "01_decomposition.png"),
    ("2. SAM masks (raw + resolved)", "02_sam_masks.png"),
    ("3. Segments (crop + mask)", "03_segments.png"),
    ("4. Colorized segments", "04_colorized_segments.png"),
    ("5. Blended result", "05_blended.png"),
]
for title, filename in stages:
    path = os.path.join(STAGES_DIR, filename)
    print(title)
    if os.path.exists(path):
        display(IPImage(filename=path))
    else:
        print("  missing:", path)

In [ ]:
# Cell 7: display the input, result, and routed domain map
import os
from pathlib import Path
from IPython.display import Image as IPImage, display

print('Input')
if os.path.exists(IMAGE_PATH):
    display(IPImage(filename=IMAGE_PATH))
else:
    print(f'  Not found: {IMAGE_PATH}')

print('Colorized')
if os.path.exists(OUTPUT_PATH):
    display(IPImage(filename=OUTPUT_PATH))
else:
    print(f'  Not found: {OUTPUT_PATH}')
    print('  Cell 5 must complete successfully first. Check its output for errors.')

print('Domain map')
domain_map = OUTPUT_PATH.replace('.png', '_domains.png')
if os.path.exists(domain_map):
    display(IPImage(filename=domain_map))
else:
    print(f'  Not found: {domain_map}')


### Notes
- `--visualize` saves the intermediate stages to `outputs/stages/`: `01_decomposition.png`, `02_sam_masks.png`, `03_segments.png`, `04_colorized_segments.png`, `05_blended.png`, plus per-region files in `outputs/stages/regions/`.
- To skip detection and colorize the whole image with the general expert, add `--skip-detection` to the command above.
- For real per-domain experts, replace `--single-checkpoint` with `--checkpoints-dir checkpoints/`; the directory should contain `people.pth`, `nature.pth`, `urban.pth`, `animals.pth`, and `fallback.pth`.
- SAM ViT-B (~375 MB) is downloaded automatically on the first run if `sam_vit_b.pth` is missing.

In [ ]:
# Cell 9: Run Person C tests (metrics, plotting, test sets, E3 baseline)
!PYTHONPATH=. python -m pytest -q tests/test_metrics.py tests/test_plotting.py tests/test_test_sets.py tests/test_e3_baseline.py

In [ ]:
# Cell 9a-new: Regenerate E3 ID lists from real val2017 files
# This cell:
#   1. Scans the actual val2017 directory for existing files
#   2. Builds 5 domain pools using real instance/stuff annotations
#   3. Saves outputs/e3_ids.json
#   4. Rewrites colorize/test_sets.py in-place so Cell 9b immediately passes
import json, random, re
from pathlib import Path
from collections import defaultdict
import numpy as np
from PIL import Image

# ── 1. Locate COCO root ──────────────────────────────────────────────────
val2017_dir = next((p for p in Path('/kaggle/input').rglob('val2017') if p.is_dir()), None)
if val2017_dir is None:
    raise FileNotFoundError('COCO val2017 not found under /kaggle/input')
ROOT = val2017_dir.parent

# ── 2. Load instance annotations ─────────────────────────────────────────
ann = json.load(open(ROOT / 'annotations' / 'instances_val2017.json'))
cats = {c['id']: c['name'] for c in ann['categories']}
imgs = {i['id']: i for i in ann['images']}
exists_val = {i for i in imgs if (val2017_dir / f'{i:012d}.jpg').exists()}
print(f'val2017 images on disk: {len(exists_val)}')

ANIMALS = {'bird','cat','dog','horse','sheep','cow','elephant','bear','zebra','giraffe'}
area = defaultdict(lambda: defaultdict(float))
for a in ann['annotations']:
    area[a['image_id']][cats[a['category_id']]] += a['area']

def inst_frac(img_id, names):
    tot = imgs[img_id]['width'] * imgs[img_id]['height']
    return sum(area[img_id][n] for n in names) / tot if tot else 0.0

people_pool  = [i for i in exists_val if inst_frac(i, {'person'}) >= 0.20 and inst_frac(i, ANIMALS) == 0]
animals_pool = [i for i in exists_val if inst_frac(i, ANIMALS) >= 0.20 and inst_frac(i, {'person'}) == 0]
print(f'people candidates  (inst>=20% person, 0 animal): {len(people_pool)}')
print(f'animals candidates (inst>=20% animal, 0 person): {len(animals_pool)}')

# ── 3. Stuff-based pools (nature, urban, mixed) ───────────────────────────
stuff_dir = next(
    (p for p in Path('/kaggle/input').rglob('stuffthingmaps_val2017') if p.is_dir()),
    next((p for p in Path('/kaggle/input').rglob('stuffthingmaps') if p.is_dir()), None)
)
labels_txt = next(Path('/kaggle/input').rglob('labels.txt'), None)
GROUPS_PATH = Path('/kaggle/working/domain-adaptive-colorization/configs/class_groups.json')

if stuff_dir is None or labels_txt is None:
    print('WARNING: COCO-Stuff stuffthingmaps or labels.txt not found.')
    print('  Nature/urban/mixed will use instance-annotation fallbacks.')
    # Fallback: nature = outdoor images with low person+animal coverage
    # (uses only instances_val2017.json — no stuff maps needed)
    OUTDOOR = {'bird','boat','airplane','horse','sheep','cow','elephant','bear',
                'zebra','giraffe','kite','surfboard','skis','snowboard','sports ball',
                'frisbee','skateboard'}
    nature_pool  = [i for i in exists_val
                    if inst_frac(i, {'person'}) < 0.05
                    and inst_frac(i, ANIMALS)   < 0.05
                    and inst_frac(i, OUTDOOR)   < 0.10]
    VEHICLES = {'bicycle','car','motorcycle','bus','train','truck'}
    urban_pool   = [i for i in exists_val
                    if inst_frac(i, VEHICLES)   >= 0.10
                    and inst_frac(i, {'person'}) < 0.05]
    mixed_pool   = [i for i in exists_val
                    if inst_frac(i, {'person'}) >= 0.10
                    and inst_frac(i, ANIMALS)   >= 0.10]
    print(f'nature candidates  (instance fallback, low person+animal): {len(nature_pool)}')
    print(f'urban  candidates  (instance fallback, vehicles>=10%):     {len(urban_pool)}')
    print(f'mixed  candidates  (instance fallback, person+animal>=10%): {len(mixed_pool)}')
else:
    raw_names = [l.strip() for l in labels_txt.read_text().splitlines() if l.strip() and not l.startswith('#')]
    label_names = []
    for l in raw_names:
        m = re.match(r'^\d+\s*:\s*(.+)$', l)
        label_names.append(m.group(1).strip() if m else l)
    groups_cfg = json.load(open(GROUPS_PATH))
    name_to_group = {}
    for grp, info in groups_cfg['groups'].items():
        for cls in info['classes']:
            name_to_group[cls.lower()] = grp
    pv_to_group = {}
    for pv in range(182):
        idx = pv + 1
        if 0 <= idx < len(label_names):
            grp = name_to_group.get(label_names[idx].lower())
            if grp:
                pv_to_group[pv] = grp
    nature_pool = []; urban_pool = []; mixed_pool = []
    for img_id in exists_val:
        png_path = stuff_dir / f'{img_id:012d}.png'
        if not png_path.exists():
            continue
        pv = np.array(Image.open(png_path))
        total = pv.size
        group_px = defaultdict(int)
        for pixel_val, grp in pv_to_group.items():
            group_px[grp] += int(np.sum(pv == pixel_val))
        def frac(grps):
            return sum(group_px[g] for g in grps) / total
        sv = frac(['sky', 'vegetation']); bld = frac(['building'])
        per = frac(['person']); ani = frac(['animal'])
        if sv >= 0.40 and (per + ani) < 0.02:
            nature_pool.append(img_id)
        if bld >= 0.25 and per < 0.05:
            urban_pool.append(img_id)
        fracs = {g: frac([g]) for g in ['person','animal','sky','vegetation','building']}
        if sum(1 for f in fracs.values() if f >= 0.10) >= 2:
            mixed_pool.append(img_id)
    print(f'nature candidates  (sky+veg>=40%, per+ani<2%): {len(nature_pool)}')
    print(f'urban  candidates  (building>=25%, per<5%):    {len(urban_pool)}')
    print(f'mixed  candidates  (>=2 groups each >=10%):    {len(mixed_pool)}')

# ── 4. Sample 20 per set, no overlap ─────────────────────────────────────
random.seed(0)
sets = {}
used = set()
POOLS = [
    ('domain_people',  people_pool),
    ('domain_nature',  nature_pool),
    ('domain_urban',   urban_pool),
    ('domain_animals', animals_pool),
    ('mixed_scene',    mixed_pool),
]
for name, pool in POOLS:
    available = sorted(set(pool) - used)
    if len(available) < 20:
        print(f'WARNING: {name} has only {len(available)} candidates -- using all available')
    sample = sorted(random.sample(available, min(20, len(available))))
    sets[name] = sample
    used |= set(sample)
    print(f'{name}: {len(available)} candidates -> sampled {len(sample)}')

Path('outputs').mkdir(exist_ok=True)
json.dump(sets, open('outputs/e3_ids.json', 'w'), indent=1)
print('\nSaved to outputs/e3_ids.json')

# ── 5. Rewrite colorize/test_sets.py in-place ────────────────────────────
# This is what makes Cell 9b immediately pass without a git push.
REPO = Path('/kaggle/working/domain-adaptive-colorization')
ts_path = REPO / 'colorize' / 'test_sets.py'
src = ts_path.read_text(encoding='utf-8')

def fmt_list(ids, indent=8):
    """Format a list of ints as a compact multi-line Python literal."""
    pad = ' ' * indent
    rows = [ids[i:i+5] for i in range(0, len(ids), 5)]
    lines = [', '.join(str(x) for x in row) for row in rows]
    return ('[\n' + pad + (',\n' + pad).join(lines) + ',\n' +
            ' ' * (indent - 4) + ']')

import re as _re

# Replace DOMAIN_SET_IDS block
new_domain_block = (
    'DOMAIN_SET_IDS: Dict[str, List[int]] = {\n'
    '    "people": ' + fmt_list(sets['domain_people']) + ',\n'
    '    "nature": ' + fmt_list(sets['domain_nature']) + ',\n'
    '    "urban":  ' + fmt_list(sets['domain_urban'])  + ',\n'
    '    "animals":' + fmt_list(sets['domain_animals']) + ',\n'
    '}'
)
src = _re.sub(
    r'DOMAIN_SET_IDS: Dict\[str, List\[int\]\] = \{.*?\}',
    new_domain_block,
    src, flags=_re.DOTALL
)

# Replace MIXED_SCENE_IDS list
new_mixed = 'MIXED_SCENE_IDS: List[int] = ' + fmt_list(sets['mixed_scene'], indent=4)
src = _re.sub(
    r'MIXED_SCENE_IDS: List\[int\] = \[.*?\]',
    new_mixed,
    src, flags=_re.DOTALL
)

ts_path.write_text(src, encoding='utf-8')
print(f'Updated {ts_path}')
print('\nNew DOMAIN_SET_IDS:')
for k, v in sets.items():
    print(f'  {k}: {v[:3]}... ({len(v)} ids)')
print('\nRun Cell 9b now — it should print PASS with 0 mismatches.')


In [ ]:
# Cell 9b: Validate the fixed COCO IDs and save an audit manifest
import importlib, json
from pathlib import Path
import colorize.test_sets

# Reload test_sets so any in-place rewrite from Cell 9a-new takes effect immediately
importlib.reload(colorize.test_sets)

# If outputs/e3_ids.json exists, update module dictionaries in memory
e3_ids_json = Path('outputs/e3_ids.json')
if e3_ids_json.is_file():
    try:
        with e3_ids_json.open('r', encoding='utf-8') as handle:
            loaded_sets = json.load(handle)
        for dom in ['people', 'nature', 'urban', 'animals']:
            k = f'domain_{dom}'
            if k in loaded_sets:
                colorize.test_sets.DOMAIN_SET_IDS[dom] = list(loaded_sets[k])
        if 'mixed_scene' in loaded_sets:
            colorize.test_sets.MIXED_SCENE_IDS = list(loaded_sets['mixed_scene'])
    except Exception as e:
        print('Note: reading e3_ids.json:', e)

from colorize.test_sets import (
    validate_coco_ids, write_coco_metadata_manifest, MIXED_SCENE_IDS
)

# Automatically find instances_val2017.json anywhere in /kaggle/input
COCO_ANNOTATIONS = next(Path('/kaggle/input').rglob('instances_val2017.json'), None)

if COCO_ANNOTATIONS is None:
    print('COCO annotations not mounted; ID validation was not run.')
else:
    print(f'Using annotations: {COCO_ANNOTATIONS}')
    validation = write_coco_metadata_manifest(
        COCO_ANNOTATIONS, 'outputs/coco_id_manifest.json'
    )
    print('COCO ID validation report saved to outputs/coco_id_manifest.json')
    overall = 'PASS' if validation['valid'] else 'FAIL'
    n_missing = len(validation['missing_image_ids'])
    total_mismatches = sum(
        len(r['mismatches']) for r in validation['domains'].values()
    )
    print(f'Overall: {overall} | missing images: {n_missing} | domain mismatches: {total_mismatches}')
    for domain, result in validation['domains'].items():
        n_mismatch = len(result['mismatches'])
        status = 'PASS' if n_mismatch == 0 else 'FAIL'
        print(f'  domain_{domain}: {status} ({n_mismatch} mismatches)')
    # mixed_scene has no instance-category rule; report val2017 presence directly
    mixed_missing = [i for i in MIXED_SCENE_IDS if i in validation['missing_image_ids']]
    mixed_found = len(MIXED_SCENE_IDS) - len(mixed_missing)
    print(f'  mixed_scene: {mixed_found}/{len(MIXED_SCENE_IDS)} found in val2017')


In [ ]:
# Cell 10: E3 - DDColor-T/L whole-image baselines on domain + mixed-scene sets
import importlib
from pathlib import Path
import experiments.e3

# Reload e3 module so new IDs from Cell 9a-new are picked up
importlib.reload(experiments.e3)
from experiments.e3 import E3_SET_NAMES, run_e3_experiment

# Automatically find the COCO val2017 directory anywhere in /kaggle/input.
val2017_dir = next((p for p in Path('/kaggle/input').rglob('val2017') if p.is_dir()), None)
if val2017_dir is None:
    raise FileNotFoundError('COCO val2017 was not found under /kaggle/input')
COCO_ROOT = val2017_dir.parent
print(f'COCO_ROOT: {COCO_ROOT}')
print(f'E3 sets: {list(E3_SET_NAMES)} (100 images expected)')

records = run_e3_experiment(
    coco_root=COCO_ROOT,
    ddcolor_path=str(DDCOLOR_DIR),
    checkpoint_t=CHECKPOINT_T,
    checkpoint_l=CHECKPOINT_L,
    output_csv='outputs/e3_results.csv',
    summary_csv='outputs/e3_summary.csv',
    manifest_csv='outputs/e3_manifest.csv',
    device='cuda',
    gt_root=COCO_ROOT,
    allow_missing=True,
    overwrite=True,
)
print(f'E3 complete: {len(records)} records written to outputs/e3_results.csv')
print('Set-level summary: outputs/e3_summary.csv')


In [ ]:
# Cell 11: Plot E3 results
from colorize.results import load_results
from colorize.plotting import plot_metric_bar, plot_metric_comparison
from pathlib import Path

results_csv = Path('outputs/e3_results.csv')
if not results_csv.exists():
    print('outputs/e3_results.csv not found. Run Cell 10 (E3 runner) first.')
else:
    records = load_results(str(results_csv))
    print(f'Loaded {len(records)} result records')

    # Compare both whole-image baselines across the complete 100-image set.
    fig = plot_metric_bar(records, metric='colorfulness',
        title='E3: Colorfulness -- DDColor-T vs DDColor-L (whole-image baseline)',
        output_path='outputs/e3_colorfulness.png')
    if fig:
        from IPython.display import Image as IPImage, display
        if Path('outputs/e3_colorfulness.png').is_file():
            display(IPImage(filename='outputs/e3_colorfulness.png'))

    # PSNR comparison -- both T and L conditions
    psnr_records = [r for r in records if r.metric == 'psnr']
    if psnr_records:
        fig2 = plot_metric_bar(records, metric='psnr',
            title='E3: PSNR -- DDColor-T vs DDColor-L (whole-image baseline)',
            output_path='outputs/e3_psnr.png')
        if fig2:
            from IPython.display import Image as IPImage, display
            if Path('outputs/e3_psnr.png').is_file():
                display(IPImage(filename='outputs/e3_psnr.png'))

    summary = Path('outputs/e3_summary.csv')
    if summary.is_file():
        import pandas as pd
        display(pd.read_csv(summary))
    else:
        print('Summary CSV not found — check outputs/e3_summary.csv')


### E3 Experiment Notes
- `ddcolor_t_whole` uses DDColor-T (256×256 input, ConvNeXt-T backbone) on the whole image.
- `ddcolor_l_whole` uses DDColor-L (512×512 input, ConvNeXt-L backbone) on the whole image.
- Cell 10 evaluates all 100 fixed images: 20 each from people, nature, urban, animals, and mixed-scene sets.
- `outputs/e3_results.csv` contains per-image metrics; `outputs/e3_summary.csv` contains per-set mean/std values; `outputs/e3_manifest.csv` records the exact files used.
- The E3 runner is also available from the command line as `python experiments/e3.py run`.